# TP1 - Gradient Methods, Line Search, and Conjugate Gradient

## Lecture Material and Results Used

- **Gradient descent**: move in the steepest descent direction $-\nabla f(x)$ with a chosen step size.
- **Smooth convex functions**: for an $L$-smooth objective, fixed steps below roughly $2/L$ are stable on quadratics.
- **Newton method**: uses curvature through $\nabla^2 f(x)^{-1}\nabla f(x)$ and can converge very quickly near a minimizer.
- **Armijo condition**: accepts a step if it produces sufficient decrease.
- **Wolfe curvature condition**: controls whether the step is long enough along the search direction.
- **Conjugate gradient**: solves symmetric positive-definite quadratic problems using conjugate directions, often much faster than plain gradient descent.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(41)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})


## 1. Rosenbrock Function

The Rosenbrock function is a classic non-convex test problem. Its curved valley exposes the limits of naive fixed-step descent.

In [ ]:
def rosenbrock(x):
    x1, x2 = x
    return 100 * (x2 - x1 ** 2) ** 2 + (1 - x1) ** 2


def rosenbrock_grad(x):
    x1, x2 = x
    return np.array([
        -400 * x1 * (x2 - x1 ** 2) - 2 * (1 - x1),
        200 * (x2 - x1 ** 2),
    ])


def rosenbrock_hessian(x):
    x1, x2 = x
    return np.array([
        [1200 * x1 ** 2 - 400 * x2 + 2, -400 * x1],
        [-400 * x1, 200],
    ])


In [ ]:
def gradient_descent(f, grad, x0, step=1e-3, n_iter=6000):
    x = x0.astype(float).copy()
    values = []
    path = []
    for _ in range(n_iter):
        values.append(f(x))
        path.append(x.copy())
        x -= step * grad(x)
    return np.array(path), np.array(values)

path_fixed, values_fixed = gradient_descent(rosenbrock, rosenbrock_grad, np.array([-1.2, 1.0]))
plt.semilogy(values_fixed)
plt.xlabel('iteration')
plt.ylabel('f(x)')
plt.title('Fixed-step gradient descent on Rosenbrock');

## 2. Armijo Backtracking

Backtracking line search adapts the step length to the local geometry instead of requiring a manually chosen constant step.

In [ ]:
def armijo_step(f, grad, x, direction, initial_step=1.0, c=1e-4, shrink=0.5):
    step = initial_step
    fx = f(x)
    slope = grad(x) @ direction
    while f(x + step * direction) > fx + c * step * slope:
        step *= shrink
    return step


def descent_with_armijo(f, grad, x0, n_iter=2500):
    x = x0.astype(float).copy()
    values, steps = [], []
    for _ in range(n_iter):
        values.append(f(x))
        direction = -grad(x)
        step = armijo_step(f, grad, x, direction)
        x += step * direction
        steps.append(step)
    return np.array(values), np.array(steps), x

values_armijo, steps_armijo, x_armijo = descent_with_armijo(rosenbrock, rosenbrock_grad, np.array([-1.2, 1.0]))
print('final x:', x_armijo.round(4), 'final f:', values_armijo[-1].round(8))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].semilogy(values_armijo)
axes[0].set_title('objective')
axes[1].plot(steps_armijo)
axes[1].set_title('accepted step sizes')
fig.tight_layout();

### Strong Wolfe line search

Armijo accepts any sufficiently small step with enough decrease. Strong Wolfe also bounds the magnitude of the new directional derivative. Compare accepted steps from the same point on the Rosenbrock function.

In [ ]:
from scipy.optimize import line_search

point = np.array([-1.2, 1.0])
direction = -rosenbrock_grad(point)
slope = rosenbrock_grad(point) @ direction
alpha = line_search(rosenbrock, rosenbrock_grad, point, direction, c2=0.1)[0]
assert alpha is not None
for label, step in [("Armijo", armijo_step(rosenbrock, rosenbrock_grad, point, direction)), ("Strong Wolfe", alpha)]:
    trial = point + step * direction
    ratio = abs(rosenbrock_grad(trial) @ direction / slope)
    print(f"{label}: step {step:.6f}, absolute slope ratio {ratio:.3f}")
assert rosenbrock(point + alpha * direction) <= rosenbrock(point) + 1e-4 * alpha * slope
assert abs(rosenbrock_grad(point + alpha * direction) @ direction) <= 0.1 * abs(slope)

## 3. Newton Direction with Line Search

Newton's method uses local curvature. On Rosenbrock, a line search is still useful because the full Newton step can be too aggressive far from the minimizer.

In [ ]:
def newton_armijo(f, grad, hess, x0, n_iter=60):
    x = x0.astype(float).copy()
    values = []
    for _ in range(n_iter):
        values.append(f(x))
        g = grad(x)
        H = hess(x)
        try:
            direction = -np.linalg.solve(H, g)
        except np.linalg.LinAlgError:
            direction = -g
        if direction @ g >= 0:
            direction = -g
        step = armijo_step(f, grad, x, direction)
        x += step * direction
    return np.array(values), x

values_newton, x_newton = newton_armijo(rosenbrock, rosenbrock_grad, rosenbrock_hessian, np.array([-1.2, 1.0]))
plt.semilogy(values_armijo[:len(values_newton)], label='gradient + Armijo')
plt.semilogy(values_newton, label='Newton + Armijo')
plt.xlabel('iteration')
plt.ylabel('f(x)')
plt.title('Curvature accelerates local convergence')
plt.legend();

## 4. Conjugate Gradient on a Quadratic

For $f(x)=\frac12 x^TQx-b^Tx$, the minimizer solves $Qx=b$. Conjugate gradient exploits the quadratic structure without forming an inverse.

In [ ]:
def make_spd_problem(n=80, condition=80):
    U, _ = np.linalg.qr(rng.normal(size=(n, n)))
    eigs = np.linspace(1, condition, n)
    Q = U @ np.diag(eigs) @ U.T
    x_star = rng.normal(size=n)
    b = Q @ x_star
    return Q, b, x_star


def conjugate_gradient(Q, b, x0=None, n_iter=None):
    n = len(b)
    x = np.zeros(n) if x0 is None else x0.copy()
    r = b - Q @ x
    p = r.copy()
    errors = []
    if n_iter is None:
        n_iter = n
    for _ in range(n_iter):
        errors.append(np.linalg.norm(r))
        Qp = Q @ p
        alpha = (r @ r) / (p @ Qp)
        x += alpha * p
        r_new = r - alpha * Qp
        beta = (r_new @ r_new) / (r @ r)
        p = r_new + beta * p
        r = r_new
    return x, np.array(errors)

Q, b, x_star = make_spd_problem()
x_cg, cg_errors = conjugate_gradient(Q, b)
_, gd_values = gradient_descent(lambda x: 0.5*x@Q@x-b@x, lambda x: Q@x-b, np.zeros_like(b), step=1/np.linalg.eigvalsh(Q).max(), n_iter=len(cg_errors))
plt.semilogy(cg_errors / cg_errors[0], label='CG residual')
plt.semilogy((gd_values - gd_values.min() + 1e-12) / (gd_values[0] - gd_values.min() + 1e-12), label='GD objective gap')
plt.xlabel('iteration')
plt.title('Conjugate gradient versus fixed-step GD')
plt.legend();